# Regime analysis — weekly pattern under social entrainment

Dedicated analysis of a period where the user is constrained by a standard 5/2 work-week pattern (compressed weeknight sleep + weekend catch-up). Hypothesis tested : does the underlying N24 free-running rhythm still leak through the social-cue entrainment, and how does the week/weekend asymmetry manifest in the NPCRA indicators?

**This notebook is shipped without executed outputs** to avoid committing personal data — run it locally against your own `data/personal/<subject_id>/activity.parquet` + `periods.yaml`.

Parameterizable : change `PERIOD_NAME` below to analyse any other period defined in `periods.yaml`.

**Five analyses** :
1. 24h activity profile — weekday vs weekend overlay
2. M10 phase distribution by day-of-week (boxplot)
3. Sleep onset / wake-up time by day-of-week (from `sleep_intervals.parquet`)
4. Tau on rolling 28-day windows (does the drift evolve within the period?)
5. Tau weekdays-only vs weekends-only (does the social cycle entrain both halves equally?)

In [ ]:
import json
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy.stats import linregress

from n24sal.io import load_periods
from n24sal.npcra import (
    bootstrap_tau_ci,
    estimate_tau,
)
from n24sal.npcra.tau import _unwrap_phases_hours, m10_phases_per_day
from n24sal.viz import apply_theme
from n24sal.viz.theme import AMBER, CYAN, TEAL

apply_theme("dark")
pio.renderers.default = "notebook_connected"

SUBJECT_ID = "S001"
PERIOD_NAME = "regime_atcf"  # change to analyse a different period from periods.yaml
DATA_DIR = Path("..") / "data" / "personal" / SUBJECT_ID
EPOCHS_PER_HOUR = 60
EPOCHS_PER_DAY = 1440
WEEKDAY_NAMES = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

In [ ]:
# Load and slice to the period
activity = pd.read_parquet(DATA_DIR / "activity.parquet")
sleep = pd.read_parquet(DATA_DIR / "sleep_intervals.parquet")
metadata = json.loads((DATA_DIR / "subject_metadata.json").read_text())
TIMEZONE = metadata["timezone"]
periods_file = load_periods(DATA_DIR / "periods.yaml")
period = periods_file.get(PERIOD_NAME)

start_local = pd.Timestamp(period.start, tz=TIMEZONE)
end_local = pd.Timestamp(period.end, tz=TIMEZONE)

activity["ts_local"] = activity["timestamp"].dt.tz_convert(TIMEZONE)
mask = (activity["ts_local"] >= start_local) & (activity["ts_local"] < end_local)
act = activity[mask].sort_values("timestamp").reset_index(drop=True)

sleep["ts_local_start"] = sleep["stage_start"].dt.tz_convert(TIMEZONE)
sleep["ts_local_end"] = sleep["stage_end"].dt.tz_convert(TIMEZONE)
smask = (sleep["ts_local_start"] >= start_local) & (sleep["ts_local_start"] < end_local)
slp = sleep[smask].sort_values("ts_local_start").reset_index(drop=True)

n_present = int(act["present"].sum()) if "present" in act.columns else len(act)
print(f"Period {PERIOD_NAME}: {period.start} → {period.end}")
print(f"  Activity dense epochs : {len(act):,}  |  present {n_present:,} ({100*n_present/len(act):.1f}%)")
print(f"  Sleep stages          : {len(slp):,}")
print(f"  Notes                 : {period.notes!r}")

## 1. Activity 24h profile — weekday vs weekend overlay

Two means superposed : Mon-Fri (work schedule) vs Sat-Sun (free schedule). If the user's circadian rhythm is *only* entrained by social cues, the weekend profile should drift right (later bedtime, later wake) compared to weekday.

In [ ]:
act["weekday"] = act["ts_local"].dt.dayofweek
act["is_weekend"] = act["weekday"] >= 5
act["min_of_day"] = act["ts_local"].dt.hour * 60 + act["ts_local"].dt.minute

prof_wd = act[~act["is_weekend"]].groupby("min_of_day")["activity"].mean().reindex(range(1440), fill_value=0)
prof_we = act[act["is_weekend"]].groupby("min_of_day")["activity"].mean().reindex(range(1440), fill_value=0)

hours = np.arange(1440) / 60
fig = go.Figure()
fig.add_trace(go.Scatter(x=hours, y=prof_wd.values, mode="lines",
                         line=dict(color=TEAL, width=2), name="weekday (Mon–Fri)"))
fig.add_trace(go.Scatter(x=hours, y=prof_we.values, mode="lines",
                         line=dict(color=AMBER, width=2), name="weekend (Sat–Sun)"))
fig.update_layout(
    title=f"24h activity profile — weekday vs weekend — {PERIOD_NAME}",
    xaxis=dict(title=f"hour ({TIMEZONE})", tick0=0, dtick=3, range=[0, 24]),
    yaxis_title="mean activity",
)
fig.show()

## 2. M10 phase distribution by day-of-week

M10 start hour per calendar day (in the period), grouped by day-of-week. If the user respects a fixed work schedule on weekdays, the Mon-Fri boxes should be tight ; weekend boxes wider (free schedule). The difference between weekday and weekend medians = the *social jet lag* in chronobiology terms.

In [ ]:
arr = act["activity"].to_numpy()
phases = m10_phases_per_day(arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY)
n_days = len(phases)

first_local_day = act["ts_local"].iloc[0].normalize()
day_dates = pd.date_range(first_local_day, periods=n_days, freq="D")
weekdays = day_dates.dayofweek
day_names = [WEEKDAY_NAMES[w] for w in weekdays]

phase_df = pd.DataFrame({
    "date": day_dates,
    "day": day_names,
    "weekday_idx": weekdays,
    "M10_phase": phases,
})

fig = px.box(
    phase_df, x="day", y="M10_phase",
    category_orders={"day": WEEKDAY_NAMES},
    color_discrete_sequence=[TEAL],
)
fig.update_layout(
    title=f"M10 phase by day-of-week — {PERIOD_NAME}",
    yaxis_title="M10 start hour (0–24)",
    xaxis_title="",
)
fig.show()

print(phase_df.groupby("day")["M10_phase"].agg(["median", "std", "count"]).round(2)
      .reindex(WEEKDAY_NAMES))

## 3. Sleep onset / wake-up time by day-of-week

Per-night aggregation via `n24sal.sleep.main_sleep_per_night` — identifies the **longest sleep session per night** (the chronobiological "main sleep period") using Samsung's `sleep_id` grouping and midpoint-based night assignment (robust to long sessions straddling the 20h boundary, and to daytime naps).

For each night :
- `main_onset` / `main_offset` describe the longest session (= main sleep)
- `main_duration_h` = duration of that session
- `tst_h` = total sleep time across **all** sessions in the night (main + naps)
- `n_sessions` = count of distinct sleep sessions in the night

Onset / wake reported as hours-from-20h to keep monotone over the night window (20h → 0, 23h → 3, 0h → 4, 7h → 11).

In [ ]:
from n24sal.sleep import main_sleep_per_night


def hours_from_20h(ts):
    h = ts.hour + ts.minute / 60
    return (h - 20) % 24


per_night = main_sleep_per_night(slp, timezone=TIMEZONE)
per_night["onset_h20"] = per_night["main_onset"].apply(hours_from_20h)
per_night["offset_h20"] = per_night["main_offset"].apply(hours_from_20h)
per_night["weekday_idx"] = pd.to_datetime(per_night.index).dayofweek
per_night["day"] = per_night["weekday_idx"].map(lambda w: WEEKDAY_NAMES[w])

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        "Sleep onset (h-from-20h)",
        "Wake-up (h-from-20h)",
        "Main sleep duration (h)",
    ),
)
for day in WEEKDAY_NAMES:
    sub = per_night[per_night["day"] == day]
    fig.add_trace(go.Box(y=sub["onset_h20"], name=day, marker_color=TEAL, showlegend=False), row=1, col=1)
    fig.add_trace(go.Box(y=sub["offset_h20"], name=day, marker_color=AMBER, showlegend=False), row=1, col=2)
    fig.add_trace(go.Box(y=sub["main_duration_h"], name=day, marker_color=CYAN, showlegend=False), row=1, col=3)
fig.update_layout(title=f"Main sleep onset / wake-up / duration by weekday — {PERIOD_NAME}", height=420)
fig.update_yaxes(range=[0, 24], row=1, col=1)
fig.update_yaxes(range=[0, 24], row=1, col=2)
fig.show()

summary = per_night.groupby("day")[["onset_h20", "offset_h20", "main_duration_h", "tst_h", "n_sessions"]].median().round(2)
print(summary.reindex(WEEKDAY_NAMES))

## 4. Tau on 28-day rolling windows

Does the linear drift evolve over the 101-day period? Tau computed on every 28-day window stepped by 7 days. R² < 0.5 → no convincing drift in that window. R² ≥ 0.85 → solid free-running signature.

In [ ]:
WINDOW_DAYS = 28
STEP_DAYS = 7
MIN_COV = 0.5

first = act["ts_local"].dt.normalize().min()
last = act["ts_local"].dt.normalize().max()

rolling = []
current = first
while current + pd.Timedelta(days=WINDOW_DAYS) <= last:
    end_w = current + pd.Timedelta(days=WINDOW_DAYS)
    wmask = (act["ts_local"] >= current) & (act["ts_local"] < end_w)
    sub = act[wmask].sort_values("timestamp")
    if len(sub) < EPOCHS_PER_DAY * WINDOW_DAYS * 0.5:
        current += pd.Timedelta(days=STEP_DAYS); continue
    sub_arr = sub["activity"].to_numpy()
    sub_mask = sub["present"].to_numpy() if "present" in sub.columns else None
    t = estimate_tau(sub_arr, EPOCHS_PER_HOUR, EPOCHS_PER_DAY,
                     present_mask=sub_mask, min_daily_coverage=MIN_COV)
    rolling.append({
        "window_start": current,
        "tau": t.tau_hours,
        "r2": t.r_squared,
        "n_days": t.n_days,
    })
    current += pd.Timedelta(days=STEP_DAYS)

rolling_df = pd.DataFrame(rolling)
print(f"{len(rolling_df)} rolling 28d windows")
print(rolling_df.round(3).to_string(index=False))

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("tau (h)", "R²"), vertical_spacing=0.08)
fig.add_trace(go.Scatter(x=rolling_df["window_start"], y=rolling_df["tau"], mode="lines+markers",
                         line=dict(color=TEAL), name="tau"), row=1, col=1)
fig.add_hline(y=24.0, line=dict(color=AMBER, dash="dot"), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling_df["window_start"], y=rolling_df["r2"], mode="lines+markers",
                         line=dict(color=CYAN), name="R²"), row=2, col=1)
fig.add_hline(y=0.85, line=dict(color=AMBER, dash="dot"), row=2, col=1)
fig.update_layout(title=f"Tau on 28d rolling windows — {PERIOD_NAME}", showlegend=False, height=520)
fig.update_xaxes(title_text="window start", row=2, col=1)
fig.show()

## 5. Tau weekdays-only vs weekends-only

M10 phases regressed on day index, separately for weekdays (Mon-Fri) and weekends (Sat-Sun). If the social schedule entrains the rhythm uniformly, both slopes should be similar (~0). If the weekend allows the rhythm to free-run, the weekend slope should be different (positive for late chronotype N24).

Bootstrap CI on each, with non-contiguous day indices preserved (we don't compact ; the slope is in hours per *calendar* day).

In [ ]:
def regress_subset(phases_arr, weekday_arr, keep_mask, label, n_boot=500, seed=42):
    sub_phases = phases_arr[keep_mask]
    sub_idx = np.where(keep_mask)[0].astype(float)  # original calendar day indices
    if len(sub_phases) < 3:
        return {"label": label, "tau": float("nan"), "r2": float("nan"),
                "ci_low": float("nan"), "ci_high": float("nan"), "n": len(sub_phases)}
    unwrapped = _unwrap_phases_hours(sub_phases)
    fit = linregress(sub_idx, unwrapped)
    # Bootstrap slope CI
    rng = np.random.default_rng(seed)
    slopes = np.empty(n_boot)
    for i in range(n_boot):
        boot = rng.integers(0, len(sub_phases), size=len(sub_phases))
        if len(np.unique(sub_idx[boot])) < 2:
            slopes[i] = np.nan; continue
        slopes[i] = linregress(sub_idx[boot], unwrapped[boot]).slope
    valid = slopes[~np.isnan(slopes)]
    ci_low, ci_high = np.percentile(valid, [2.5, 97.5])
    return {
        "label": label,
        "tau": 24.0 + fit.slope,
        "r2": fit.rvalue**2,
        "ci_low": 24.0 + ci_low,
        "ci_high": 24.0 + ci_high,
        "n": len(sub_phases),
    }

weekdays_mask = weekdays < 5
weekends_mask = weekdays >= 5

results = [
    regress_subset(phases, weekdays, np.ones(n_days, dtype=bool), "all days"),
    regress_subset(phases, weekdays, weekdays_mask, "weekdays Mon–Fri"),
    regress_subset(phases, weekdays, weekends_mask, "weekends Sat–Sun"),
]
print(pd.DataFrame(results).round(4).to_string(index=False))

In [ ]:
# Visualize: scatter M10 phases color-coded by weekday status, with regression lines
fig = go.Figure()
wd_idx = np.where(weekdays_mask)[0]
we_idx = np.where(weekends_mask)[0]
fig.add_trace(go.Scatter(x=wd_idx, y=phases[wd_idx], mode="markers",
                         marker=dict(color=TEAL, size=6), name="weekday"))
fig.add_trace(go.Scatter(x=we_idx, y=phases[we_idx], mode="markers",
                         marker=dict(color=AMBER, size=6), name="weekend"))
# Regression lines — use raw phases for visual, even if unwrap was used numerically
for mask, color, name in [(weekdays_mask, TEAL, "weekday fit"), (weekends_mask, AMBER, "weekend fit")]:
    sub_phases = phases[mask]
    sub_idx = np.where(mask)[0].astype(float)
    if len(sub_phases) >= 3:
        unwrapped = _unwrap_phases_hours(sub_phases)
        fit = linregress(sub_idx, unwrapped)
        line_x = np.array([sub_idx.min(), sub_idx.max()])
        line_y = (fit.intercept + fit.slope * line_x) % 24
        fig.add_trace(go.Scatter(x=line_x, y=fit.intercept + fit.slope * line_x,
                                 mode="lines", line=dict(color=color, dash="dot"),
                                 name=f"{name} (slope={fit.slope:+.3f}h/d)"))
fig.update_layout(
    title=f"M10 phase by day index — weekday vs weekend regression — {PERIOD_NAME}",
    xaxis_title="day index from period start",
    yaxis_title="M10 phase (h)",
)
fig.show()

## Key results — to fill in after running

**Analyse 1 — profile 24h** : décalage du pic d'activité weekend vs semaine = ___ h. Cohérent avec un retard de phase social ("social jet lag") les weekends ?

**Analyse 2 — M10 phase par weekday** : médiane weekday = ___ h vs weekend = ___ h. Écart = ___ h. IQR weekday = [___, ___], weekend = [___, ___].

**Analyse 3 — sleep onset / wake** : onset médian semaine = ___ h-from-20h vs weekend = ___ h-from-20h. Durée semaine = ___ h vs weekend = ___ h.

**Analyse 4 — tau rolling 28d** : tau oscille entre ___ h et ___ h sur les fenêtres. R² maximum = ___ atteint sur la fenêtre commençant le ___ . Pas de tendance monotone visible / tendance monotone (drift accélère / ralentit).

**Analyse 5 — tau weekday vs weekend** : weekday-only tau = ___ h (R²=___), weekend-only tau = ___ h (R²=___). Si weekend-only > weekday-only : confirmation de l'entraînement social masking le free-running. Si égaux ≈ 24h : la régularité de la semaine est interne, pas externe.

*Cette section devient le texte de la sous-partie "Social entrainment" du manuscrit case-report.*